# Day 3, Lesson 3.4: The LangGraph agent (3 tools)

## Step 1: The 3 tools

In [1]:
import json
import sys

sys.path.insert(0, "../src")
from langchain_core.utils.function_calling import convert_to_openai_tool

from agent.tools import TOOLS

for t in TOOLS:
    schema = convert_to_openai_tool(t)["function"]
    print(f"🔧 {schema['name']}")
    print(f"   description: {schema['description'][:110]}...")
    print(f"   parameters:  {json.dumps(schema['parameters'].get('properties', {}))}\n")

🔧 get_live_weather
   description: LIVE Mumbai weather right now: temperature, feels-like, humidity, rain, clouds, wind and condition,
    plus O...
   parameters:  {}

🔧 predict_next_hour_temperature
   description: Our own trained ML model (Day 1, gradient boosting): predicts Mumbai's temperature for the NEXT HOUR
    from ...
   parameters:  {}

🔧 search_imd_documents
   description: Search the official IMD (India Meteorological Department) reference documents: heat wave and cold wave
criteri...
   parameters:  {"query": {"description": "what to look up, written as a short question, e.g. \"heat wave criteria for coastal stations\"", "type": "string"}}



## Step 2: Build the graph, step by step

In [2]:
import logging

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from agent.graph import AGENT_RULES, _make_llm

logging.basicConfig(level=logging.WARNING)
print("THE AGENT'S RULES:\n" + AGENT_RULES)

llm = _make_llm()

def agent_node(state: MessagesState) -> dict:
    reply = llm.invoke([SystemMessage(AGENT_RULES)] + state["messages"])
    return {"messages": [reply]}

graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(TOOLS))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition)
graph.add_edge("tools", "agent")
agent = graph.compile()

print("\nTHE GRAPH (Mermaid diagram code; paste it into https://mermaid.live to see it drawn):")
print(agent.get_graph().draw_mermaid())

THE AGENT'S RULES:
You are a weather assistant for operations staff in Mumbai, India.

Which tool to use:
- Weather right now, or rain/conditions over the next hours: get_live_weather.
- Temperature in the NEXT HOUR: predict_next_hour_temperature (our own trained model). Say it is our model's prediction.
- Definitions, thresholds, rules, warning levels or official advice: search_imd_documents.
- A question may need more than one tool (e.g. "is the current temperature a heat wave?" needs live data AND the IMD criteria).
- Greetings or questions about what you can do: answer directly, no tool.

Rules:
- You only cover Mumbai. For other places, say so.
- Use only facts from tool results. Never invent numbers or rules.
- When you use IMD document passages, put their chunk id in square brackets after the fact, e.g. [heat_faq-004].
- If a tool returns an error or finds nothing, say the information is unavailable.
- Be short and plain: 1-4 sentences.



THE GRAPH (Mermaid diagram code; paste it into https://mermaid.live to see it drawn):
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 3: Watch one question, node by node

In [3]:
question = "who is PM of India"
print("QUESTION:", question, "\n")

for update in agent.stream({"messages": [HumanMessage(question)]},
                           config={"recursion_limit": 8}, stream_mode="updates"):
    for node, change in update.items():
        for msg in change["messages"]:
            if node == "agent" and msg.tool_calls:
                print(f"[agent] asks for tools: {[(c['name'], c['args']) for c in msg.tool_calls]}")
            elif node == "agent":
                print(f"[agent] final answer: {msg.text.strip()}")
            else:
                print(f"[tools] {msg.name} returned {len(msg.content)} characters: {msg.content[:110]}...")

QUESTION: who is PM of India 



[agent] final answer: I am a weather assistant for operations staff in Mumbai and can only answer questions about Mumbai's weather, forecasts, and official meteorological guidelines. I cannot provide information about political figures such as the Prime Minister of India.


## Step 4: Routing test: the right tool for each kind of question

In [4]:
import pandas as pd

from agent import run_agent

ROUTING_TESTS = [
    ("How hot is it in Mumbai right now?", {"get_live_weather"}),
    ("What will the temperature be in Mumbai in one hour?", {"predict_next_hour_temperature"}),
    ("What counts as very heavy rainfall according to IMD?", {"search_imd_documents"}),
    ("Is the current temperature in Mumbai a heat wave by IMD rules?", {"get_live_weather", "search_imd_documents"}),
    ("Hi! What can you help me with?", set()),
    ("What's the weather like in Delhi?", set()),
]

rows = []
for question, expected in ROUTING_TESTS:
    result = run_agent(question)
    used = set(result["tools_used"])
    rows.append({"question": question, "expected": ", ".join(sorted(expected)) or "(none)",
                 "used": ", ".join(sorted(used)) or "(none)",
                 "PASS": "✅" if used == expected else "❌"})
    print(f"Q: {question}\n   tools: {result['tools_used'] or 'none'}\n   answer: {result['answer']}")
    for s in result["sources"]:
        print(f"   source: [{s['chunk_id']}] {s['document']}, pages {s['pages']}")
    print()

routing = pd.DataFrame(rows)
print(f"Routing: {(routing['PASS'] == '✅').sum()} of {len(routing)} correct")
routing

Q: How hot is it in Mumbai right now?
   tools: ['get_live_weather']
   answer: Right now in Mumbai, the temperature is 35.8°C with a feels-like temperature of 38.4°C under mainly clear skies.



Q: What will the temperature be in Mumbai in one hour?
   tools: ['predict_next_hour_temperature']
   answer: Based on our trained ML model, the predicted temperature in Mumbai for the next hour is 34.7°C (a decrease of 0.7°C from the last observed temperature of 35.4°C).



Q: What counts as very heavy rainfall according to IMD?
   tools: ['search_imd_documents']
   answer: According to the IMD, very heavy rainfall is defined as 24-hour accumulated rainfall ranging from 115.6 mm to 204.4 mm (12 cm to 20 cm) [forecast_sop-168].
   source: [forecast_sop-168] IMD: SOP for Weather Forecasting and Warning, pages 24-25



Q: Is the current temperature in Mumbai a heat wave by IMD rules?
   tools: ['get_live_weather', 'search_imd_documents']
   answer: Mumbai's current temperature is 35.8°C (with today's maximum reaching 36°C). By IMD rules for coastal stations, a heat wave requires an actual maximum temperature of 37°C or more along with a departure of 4.5°C or more from normal [heat_faq-004]. Therefore, the current temperature does not meet the criteria for a heat wave.
   source: [heat_faq-004] IMD: FAQ on Heat Wave, pages 2-2



Q: Hi! What can you help me with?
   tools: none
   answer: I can help you with live weather conditions and short-term forecasts for Mumbai, temperature predictions for the next hour using our ML model, and official IMD criteria, definitions, and warning guidelines from reference documents.



Q: What's the weather like in Delhi?
   tools: none
   answer: I only cover Mumbai, so I cannot provide weather information for Delhi.

Routing: 6 of 6 correct


,question,expected,used,PASS
0,How hot is it in Mumbai right now?,get_live_weather,get_live_weather,✅
1,What will the temperature be in Mumbai in one ...,predict_next_hour_temperature,predict_next_hour_temperature,✅
2,What counts as very heavy rainfall according t...,search_imd_documents,search_imd_documents,✅
3,Is the current temperature in Mumbai a heat wa...,"get_live_weather, search_imd_documents","get_live_weather, search_imd_documents",✅
4,Hi! What can you help me with?,(none),(none),✅
5,What's the weather like in Delhi?,(none),(none),✅
